# 01 — Data Collection & Understanding
## FMD Outbreak Prediction — Sri Lanka (2017–2024)

**Objective:** This notebook introduces all the raw data sources used in this
research project. Each dataset is loaded and explored so we can understand
its structure, coverage, and quality before any processing.

### Data Sources
| # | Source | Description | Resolution |
|---|--------|-------------|------------|
| 1 | DAPH | FMD outbreak records from Department of Animal Production & Health | Monthly, by district |
| 2 | CHIRPS | Rainfall estimates from Climate Hazards Group | Dekadal → monthly, by district |
| 3 | GLW | Gridded Livestock of the World (buffalo & cattle density) | Spatial grid, 2015 & 2020 |
| 4 | Admin Boundaries | Sri Lanka district shapefiles (OCHA/HDX) | District level (admin2) |
| 5 | Weather | Temperature, humidity, wind speed (external) | Monthly, by district |

---
## 1. FMD Outbreak Records (DAPH)

The Department of Animal Production & Health (DAPH) of Sri Lanka maintains
records of Foot-and-Mouth Disease outbreaks across all 25 districts.
This is our **target variable** — whether an outbreak occurred in a given
district-month.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

BASE_DIR = r'D:\Projects\Research_Component'
RAW_DIR  = os.path.join(BASE_DIR, 'data', 'raw')
PLOT_DIR = os.path.join(BASE_DIR, 'plots', '01_data_overview')
os.makedirs(PLOT_DIR, exist_ok=True)

# Load DAPH outbreak data
daph_file = os.path.join(RAW_DIR, 'daph', 'FMD_SriLanka_2017_2024 (without case counts) - usable.xlsx')
daph = pd.read_excel(daph_file)

print(f"Shape: {daph.shape}")
print(f"Columns: {daph.columns.tolist()}")
print(f"\nData types:\n{daph.dtypes}")
print(f"\nFirst 5 rows:")
daph.head()

### Key observations — DAPH data
- **2400 rows** = 25 districts × 8 years (2017–2024) × 12 months
- **Target column:** `Outbreak status` (0 = no outbreak, 1 = outbreak)
- No missing values expected — this is a complete monthly panel

In [ ]:
# Basic statistics
print(f"Year range: {daph['Year'].min()} – {daph['Year'].max()}")
print(f"Districts: {daph['District'].nunique()} → {sorted(daph['District'].unique())}")
print(f"\nOutbreak status distribution:")
print(daph['Outbreak status'].value_counts())
print(f"\nMissing values:\n{daph.isnull().sum()}")

---
## 2. Rainfall Data (CHIRPS)

CHIRPS (Climate Hazards Group InfraRed Precipitation with Station data)
provides high-resolution rainfall estimates. The data is at **dekadal**
(10-day) resolution and will be aggregated to monthly during preprocessing.

Key rainfall features:
- `rfh` — Total rainfall (mm) for the dekad
- `r1h` — 1-day rainfall maximum
- `r3h` — 3-day rainfall maximum
- `rfq` — Rainfall frequency (number of rainy days)

In [ ]:
# Load CHIRPS rainfall data
chirps_file = os.path.join(RAW_DIR, 'chirps', 'Rainfall data from 1981 - 2026.csv')
chirps = pd.read_csv(chirps_file)

print(f"Shape: {chirps.shape}")
print(f"Columns: {chirps.columns.tolist()}")
print(f"\nData types:\n{chirps.dtypes}")
print(f"\nFirst 5 rows:")
chirps.head()

In [ ]:
# Filter to our study period (2017–2024)
chirps['date'] = pd.to_datetime(chirps['date'], format='mixed', dayfirst=True)
chirps_study = chirps[(chirps['date'].dt.year >= 2017) & (chirps['date'].dt.year <= 2024)]
print(f"\nFiltered to 2017–2024: {chirps_study.shape[0]} rows")
print(f"Date range: {chirps_study['date'].min()} to {chirps_study['date'].max()}")
print(f"Unique PCODEs: {chirps_study['PCODE'].nunique()}")
print(f"\nBasic stats for rainfall (rfh):")
print(chirps_study['rfh'].describe())

---
## 3. Livestock Density (GLW)

The Gridded Livestock of the World (GLW) dataset provides spatial
density estimates for different livestock species. We use **buffalo**
and **cattle** density data as they are the primary hosts for FMD.

Data is available for **2015** and **2020** reference years.

In [ ]:
# Check what GLW data files are available
glw_dir = os.path.join(RAW_DIR, 'glw')
for animal in ['buffolo', 'cattle']:
    animal_dir = os.path.join(glw_dir, animal)
    for year_dir in os.listdir(animal_dir):
        year_path = os.path.join(animal_dir, year_dir)
        if os.path.isdir(year_path):
            files = os.listdir(year_path)
            print(f"\n{animal}/{year_dir}: {len(files)} files")
            for f in files[:5]:
                print(f"  - {f}")
            if len(files) > 5:
                print(f"  ... and {len(files)-5} more")

> **Note:** GLW data is in raster (GeoTIFF) format. During preprocessing
> (Notebook 02), we extract district-level mean density values by overlaying
> the raster with Sri Lanka's admin boundary shapefiles.

---
## 4. Administrative Boundaries

Sri Lanka's district boundaries (shapefiles) from OCHA/HDX are used to:
- Map PCODE identifiers to district names
- Extract zonal statistics from raster datasets (GLW, weather)
- Generate spatial visualizations

In [ ]:
# Check available shapefiles
admin_dir = os.path.join(RAW_DIR, 'lka_admin_boundaries')
shp_files = [f for f in os.listdir(admin_dir) if f.endswith('.shp')]
print("Available shapefiles:")
for f in shp_files:
    print(f"  - {f}")

# Try loading district-level boundaries (admin2 = district level in Sri Lanka)
try:
    import geopandas as gpd
    districts = gpd.read_file(os.path.join(admin_dir, 'lka_admin2.shp'))
    print(f"\nDistrict boundaries loaded: {len(districts)} features")
    print(f"Columns: {districts.columns.tolist()}")
    print(districts[['ADM2_EN', 'ADM2_PCODE']].head(10))
except ImportError:
    print("\n⚠️ geopandas not installed. Install with: pip install geopandas")
    print("   Shapefiles will be used in Notebook 02 for spatial processing.")

---
## 5. Weather Data (Temperature, Humidity, Wind Speed)

Monthly weather variables (temperature, humidity, wind speed) were collected
per district. These are already incorporated into the processed intermediate
files. The original source data was obtained from meteorological records.

In [ ]:
# Check intermediate processed files to verify weather data presence
proc_dir = os.path.join(BASE_DIR, 'data', 'processed', 'while Processing')
proc_files = os.listdir(proc_dir)
print("Intermediate processed files:")
for f in proc_files:
    fpath = os.path.join(proc_dir, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f"  - {f} ({size_mb:.2f} MB)")

# Load the merged intermediate file to check weather columns
merged = pd.read_csv(os.path.join(proc_dir, 'merged_final.csv'))
weather_cols = ['temperature', 'humidity', 'wind_speed', 'temp_lag1', 'humidity_lag1', 'wind_lag1']
available = [c for c in weather_cols if c in merged.columns]
print(f"\nWeather columns in merged data: {available}")
print(f"\nWeather stats:")
print(merged[available].describe())

---
## Summary

| Data Source | Records | Key Columns | Status |
|-------------|---------|-------------|--------|
| DAPH (Outbreaks) | 2,400 | Year, District, Month, Outbreak status | ✅ Ready |
| CHIRPS (Rainfall) | ~100K+ | date, PCODE, rfh, r1h, r3h, rfq | ✅ Ready (needs monthly agg) |
| GLW (Livestock) | Raster | Buffalo & cattle density grids | ✅ Ready (needs zonal stats) |
| Admin Boundaries | 25 districts | District names, PCODEs, geometry | ✅ Ready |
| Weather | 2,400 | temperature, humidity, wind_speed | ✅ Already in processed files |

**Next:** Notebook 02 will merge all these sources into a single model-ready dataset.